In [ ]:
# ============================================================================
# Standard Library Imports
# ============================================================================
import random
import pprint

# ============================================================================
# Third-Party Library Imports
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================================
# Scikit-Learn Core Imports
# ============================================================================
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# ============================================================================
# Scikit-Learn Preprocessing Imports
# ============================================================================
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# ============================================================================
# Scikit-Learn Metrics Imports
# ============================================================================
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# ============================================================================
# Scikit-Learn Model Imports
# ============================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neural_network import MLPClassifier  

# ============================================================================
# Constants and Configuration
# ============================================================================
RANDOM_STATE = 1776
TEST_SIZE = 0.2

# Source - https://stackoverflow.com/a/9031848
# Posted by astrofrog, modified by community. See post 'Timeline' for change history
# Retrieved 2026-06-26, License - CC BY-SA 4.0

import warnings
warnings.filterwarnings('ignore')

In [714]:
df = pd.read_csv("spambase_csv.csv")

In [715]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4601 entries, 0 to 4600
Data columns (total 58 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   word_freq_make              4601 non-null   float64
 1   word_freq_address           4601 non-null   float64
 2   word_freq_all               4601 non-null   float64
 3   word_freq_3d                4601 non-null   float64
 4   word_freq_our               4601 non-null   float64
 5   word_freq_over              4601 non-null   float64
 6   word_freq_remove            4601 non-null   float64
 7   word_freq_internet          4601 non-null   float64
 8   word_freq_order             4601 non-null   float64
 9   word_freq_mail              4601 non-null   float64
 10  word_freq_receive           4601 non-null   float64
 11  word_freq_will              4601 non-null   float64
 12  word_freq_people            4601 non-null   float64
 13  word_freq_report            4601 non-null   

In [716]:
NUMBER_OF_FEATURES = len(df.columns)

In [717]:
def print_basic_metrics(model_name: str, y_test, y_pred) -> None:
    """
    Print comprehensive classification performance metrics for a binary classification model.
    
    This function calculates and displays key evaluation metrics including accuracy,
    precision, recall, F1-score, confusion matrix, and a detailed classification report.
    All metrics are printed in a formatted, readable output.
    
    Parameters
    ----------
    model_name : str
        The name or identifier of the model being evaluated. This will be displayed
        in uppercase in the output header.
    y_test : array-like of shape (n_samples,)
        True labels for the test dataset. Must be a 1D array or list containing
        the ground truth binary class labels.
    y_pred : array-like of shape (n_samples,)
        Predicted labels from the model. Must be a 1D array or list containing
        the predicted binary class labels. Should have the same length as y_test.
    
    Returns
    -------
    None
        This function prints the metrics directly to the console and does not
        return any value.
    
    Notes
    -----
    - Precision, recall, and F1-score are calculated using weighted averaging
      to account for class imbalance.
    - The confusion matrix is printed with explicit labels for each cell
      (TN, FP, FN, TP) for easy interpretation.
    - A complete classification report from scikit-learn is also displayed,
      showing per-class metrics.
    
    Examples
    --------
    >>> from sklearn.datasets import make_classification
    >>> from sklearn.model_selection import train_test_split
    >>> from sklearn.ensemble import RandomForestClassifier
    >>> 
    >>> X, y = make_classification(n_samples=100, random_state=42)
    >>> X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
    >>> model = RandomForestClassifier(random_state=42)
    >>> model.fit(X_train, y_train)
    >>> y_pred = model.predict(X_test)
    >>> 
    >>> print_basic_metrics("Random Forest", y_test, y_pred)
    ==================================================
    RANDOM FOREST PERFORMANCE METRICS
    ==================================================
    Accuracy:                0.920
    Precision (weighted):    0.921
    Recall (weighted):       0.920
    F1-Score (weighted):     0.920
    ==================================================
    
    Confusion Matrix:
    -----------------
    True Negatives:  12
    False Positives: 1
    False Negatives: 1
    True Positives:  11
    ==================================================
    
    Classification Report:
                  precision    recall  f1-score   support
    
               0       0.92      0.92      0.92        13
               1       0.92      0.92      0.92        12
    
        accuracy                           0.92        25
       macro avg       0.92      0.92      0.92        25
    weighted avg       0.92      0.92      0.92        25
    """
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)

    print("="*50)
    print(f"{model_name.upper()} PERFORMANCE METRICS")
    print("="*50)
    print(f"Accuracy:                {accuracy:.3f}")
    print(f"Precision (weighted):    {precision:.3f}")
    print(f"Recall (weighted):       {recall:.3f}")
    print(f"F1-Score (weighted):     {f1:.3f}")
    print("="*50)

    print("\nConfusion Matrix:")
    print("-----------------")
    print(f"True Negatives:  {cm[0,0]}")
    print(f"False Positives: {cm[0,1]}")
    print(f"False Negatives: {cm[1,0]}")
    print(f"True Positives:  {cm[1,1]}")
    print("="*50)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

In [718]:
classifier_metrics = dict()

### Logistic Regression

In [719]:
LOGREGRESSION_MODEL_NAME = "Logistic Regression"

In [720]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

### Logistic Regression Pipeline Implementation

In [721]:
logistic_regression = LogisticRegression(
    C=0.7,
    solver="lbfgs",
    max_iter=5000,
    l1_ratio=0,
)


num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_features),
    ("cat", cat_pipeline, categorical_features)
])

logistic_regression_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("logregression_classifier", logistic_regression)
])

logistic_regression_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('logregression_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diffe

In [722]:
y_pred = logistic_regression_pipeline.predict(X_test)
logregression_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{LOGREGRESSION_MODEL_NAME} Classifier"] = {
    "accuracy": logregression_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{LOGREGRESSION_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

LOGISTIC REGRESSION CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.914
Precision (weighted):    0.925
Recall (weighted):       0.914
F1-Score (weighted):     0.914

Confusion Matrix:
-----------------
True Negatives:  508
False Positives: 27
False Negatives: 52
True Positives:  334

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       535
           1       0.93      0.87      0.89       386

    accuracy                           0.91       921
   macro avg       0.92      0.91      0.91       921
weighted avg       0.91      0.91      0.91       921



#### Search for the Best Combination of Hyperparameters for Logistic Regression Classifier

In [723]:
param_grid = {
    "logregression_classifier__C": np.linspace(start=0.001, stop=1, num=30),
    "logregression_classifier__solver": ['lbfgs', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
    "logregression_classifier__max_iter": [num for num in range(100, 5000, 250)]
}

random_logistic_regression = RandomizedSearchCV(
    logistic_regression_pipeline,
    param_distributions=param_grid,
    n_iter=10,                       # NOTE: Number of random combinations
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_logistic_regression.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=5000))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'logregression_classifier__C': array([0.001 ..., 1. ]), 'logregression_classifier__max_iter': [100, 350, ...], 'logregression_classifier__solver': ['lbfgs', 'newton-cg', ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscros

In [724]:
best_logreg = random_logistic_regression.best_estimator_  
classifier = best_logreg.named_steps["logregression_classifier"]

print("Best Parameters:", random_logistic_regression.best_params_)
print("Best CV Score:", random_logistic_regression.best_score_)

y_pred = best_logreg.predict(X_test)
tuned_logreg_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {LOGREGRESSION_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'logregression_classifier__solver': 'newton-cholesky', 'logregression_classifier__max_iter': 4350, 'logregression_classifier__C': np.float64(0.8966551724137931)}
Best CV Score: 0.9228260869565217
TUNED LOGISTIC REGRESSION CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.913
Precision (weighted):    0.920
Recall (weighted):       0.913
F1-Score (weighted):     0.913

Confusion Matrix:
-----------------
True Negatives:  506
False Positives: 29
False Negatives: 51
True Positives:  335

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       535
           1       0.92      0.87      0.89       386

    accuracy                           0.91       921
   macro avg       0.91      0.91      0.91       921
weighted avg       0.91      0.91      0.91       921



In [725]:
classifier_metrics[f"Tuned {LOGREGRESSION_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_logreg_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
                                          'no_features': 58}}


#### Optimizing Logistic Regression Classifier

In [726]:
importances = np.abs(classifier.coef_[0])
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

word_freq_george              3.783025
word_freq_hp                  2.334188
word_freq_cs                  1.618136
char_freq_%24                 1.397661
word_freq_meeting             1.388354
word_freq_hpl                 1.181199
word_freq_85                  1.101690
word_freq_edu                 1.055998
word_freq_project             1.019115
word_freq_lab                 1.011156
word_freq_remove              0.957975
capital_run_length_longest    0.936001
word_freq_re                  0.889254
word_freq_conference          0.808751
word_freq_000                 0.806284
word_freq_free                0.802729
char_freq_%23                 0.763779
word_freq_3d                  0.747370
word_freq_credit              0.453785
word_freq_data                0.441557
word_freq_business            0.440138
capital_run_length_total      0.391154
word_freq_our                 0.389841
capital_run_length_average    0.373833
word_freq_technology          0.338398
char_freq_%3B            

In [727]:
important_features_scores = feature_importance[feature_importance > 0.9]
important_features = feature_importance[feature_importance > 0.9].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

word_freq_george              3.783025
word_freq_hp                  2.334188
word_freq_cs                  1.618136
char_freq_%24                 1.397661
word_freq_meeting             1.388354
word_freq_hpl                 1.181199
word_freq_85                  1.101690
word_freq_edu                 1.055998
word_freq_project             1.019115
word_freq_lab                 1.011156
word_freq_remove              0.957975
capital_run_length_longest    0.936001
dtype: float64
NUMBER OF FEATURES: 12


In [728]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

In [729]:
# Best Parameters: 
# {'logregression_classifier__solver': 'newton-cholesky', 
# 'logregression_classifier__max_iter': 4350, 
# 'logregression_classifier__C': np.float64(0.8966551724137931)}


In [730]:
logistic_regression = LogisticRegression(
    C=0.8966551724137931,  
    solver='newton-cholesky',  
    max_iter=4350,  
    random_state=RANDOM_STATE,
    n_jobs=-1  
)

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_features),
    ("cat", cat_pipeline, categorical_features)
])

logistic_regression_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("logregression_classifier", logistic_regression)
])

logistic_regression_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('logregression_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diffe

In [731]:
y_pred = logistic_regression_pipeline.predict(X_test)
logregression_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {LOGREGRESSION_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED LOGISTIC REGRESSION CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.863
Precision (weighted):    0.919
Recall (weighted):       0.863
F1-Score (weighted):     0.860

Confusion Matrix:
-----------------
True Negatives:  510
False Positives: 25
False Negatives: 101
True Positives:  285

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.95      0.89       535
           1       0.92      0.74      0.82       386

    accuracy                           0.86       921
   macro avg       0.88      0.85      0.85       921
weighted avg       0.87      0.86      0.86       921



In [732]:
classifier_metrics[f"Tuned and Optimized {LOGREGRESSION_MODEL_NAME} Classifier"] = {
    "accuracy": logregression_accuracy,
    "no_features": len(important_features_scores)
}

In [733]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Logistic Regression Classifier',
  {'accuracy': 0.9142236699239956, 'no_features': 58}),
 ('Tuned Logistic Regression Classifier',
  {'accuracy': 0.9131378935939196, 'no_features': 58}),
 ('Tuned and Optimized Logistic Regression Classifier',
  {'accuracy': 0.8631921824104235, 'no_features': 12})]


#### Endnotes

The narrative of the Logistic Regression model’s evolution is one of deliberate simplification, where the pursuit of parsimony was weighed against the imperative of predictive fidelity. The initial model, a standard classifier, operated on a feature space of fifty-eight dimensions, achieving a baseline accuracy of 0.9142. A subsequent tuning phase retained the full set of fifty-eight features, yielding a marginally adjusted accuracy of 0.9131, a negligible decrement that suggests the hyperparameter adjustments did little to perturb the model’s fit on this particular dataset. The most radical transformation occurred in the final iteration: a tuned and optimized version that drastically reduced the feature count to only twelve variables, a reduction of nearly eighty percent. This compression, however, came at a significant cost, as its accuracy fell to 0.8632, marking a substantial drop relative to its predecessors.

This trajectory illuminates the classic bias-variance dilemma. The first two models, with their expansive feature sets, operate in a lower-bias regime, capturing intricate patterns within the data; their high accuracy implies that this complexity did not induce prohibitive variance, at least as measured by the test metric. Conversely, the optimized model, by aggressively pruning features, deliberately increased bias in exchange for lower variance and greater generalizability. Yet, in this instance, the tradeoff proved detrimental; the degradation in accuracy—a decline of over five percentage points from the original—indicates that the sacrificed features contained critical predictive signal. The reduction in complexity was therefore not justified by performance, as the simpler model failed to offer a competitive alternative. When comparing the best performer, the base Logistic Regression, against its derivatives, it unequivocally outperforms the tuned version by a slight margin and the optimized version by a wide margin, confirming that the original configuration provided the most effective balance of predictive power and structural integrity for this task.

### KNN

In [734]:
KNN_MODEL_NAME = "K-Nearest Neighbors"

In [735]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

In [736]:
knn_base_classifier = KNeighborsClassifier(
    n_neighbors=5,
    weights="distance",
    metric="minkowski",
    p=2
)

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_features),
    ("cat", cat_pipeline, categorical_features)
])

knn_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("knn_classifier", KNeighborsClassifier())
])

knn_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('knn_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [737]:
y_pred = knn_pipeline.predict(X_test)
knn_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{KNN_MODEL_NAME} Classifier"] = {
    "accuracy": knn_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{KNN_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

K-NEAREST NEIGHBORS CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.894
Precision (weighted):    0.916
Recall (weighted):       0.894
F1-Score (weighted):     0.893

Confusion Matrix:
-----------------
True Negatives:  506
False Positives: 29
False Negatives: 69
True Positives:  317

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91       535
           1       0.92      0.82      0.87       386

    accuracy                           0.89       921
   macro avg       0.90      0.88      0.89       921
weighted avg       0.90      0.89      0.89       921



#### Search for the Best Combination of Hyperparameters for KNN Classifier

In [738]:
param_grid = {
    "knn_classifier__n_neighbors": list(range(3, 25)),
    "knn_classifier__weights": ["uniform", "distance"],
    "knn_classifier__p": [1, 2],
    "knn_classifier__algorithm": ['auto', 'ball_tree', 'kd_tree', 'brute']
}

random_knn = RandomizedSearchCV(
    knn_pipeline,
    param_distributions=param_grid,
    n_iter=5,                   # NOTE: Number of random combinations
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42
)

random_knn.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'knn_classifier__algorithm': ['auto', 'ball_tree', ...], 'knn_classifier__n_neighbors': [3, 4, ...], 'knn_classifier__p': [1, 2], 'knn_classifier__weights': ['uniform', 'distance']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variou

In [739]:
best_knn = random_knn.best_estimator_  
classifier = best_knn.named_steps["knn_classifier"]

print("Best Parameters:", random_knn.best_params_)
print("Best CV Score:", random_knn.best_score_)

y_pred = best_knn.predict(X_test)
tuned_knn_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {KNN_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'knn_classifier__weights': 'distance', 'knn_classifier__p': 2, 'knn_classifier__n_neighbors': 16, 'knn_classifier__algorithm': 'auto'}
Best CV Score: 0.9184782608695652
TUNED K-NEAREST NEIGHBORS CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.915
Precision (weighted):    0.928
Recall (weighted):       0.915
F1-Score (weighted):     0.915

Confusion Matrix:
-----------------
True Negatives:  509
False Positives: 26
False Negatives: 52
True Positives:  334

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       535
           1       0.93      0.87      0.90       386

    accuracy                           0.92       921
   macro avg       0.92      0.91      0.91       921
weighted avg       0.92      0.92      0.91       921



In [740]:
classifier_metrics[f"Tuned {KNN_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_knn_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
                                          'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
                                          'no_features': 58},
 'Tuned and Optimized Logistic Regression Classifier': {'accuracy': 0.8631921824104235,
                                                        'no_features': 12}}


#### Endnotes

The examination of the K‑Nearest Neighbors classifier proceeds through two principal versions, each operating on an identical feature space. The initial K‑Nearest Neighbors model, trained on fifty‑eight features, yields a baseline accuracy of 0.8936. Subsequently, the tuned counterpart retains precisely the same number of features—fifty‑eight—yet achieves an improved accuracy of 0.9153. This increment of roughly 2.2 percentage points arises exclusively from hyperparameter optimisation, as the feature count remains unchanged. Consequently, the architectural complexity, measured by the dimensionality of the input space, is held constant across both stages, allowing a focused evaluation of parameter fine‑tuning rather than feature reduction.

In the context of the bias‑variance tradeoff, the present scenario is atypical because model complexity is not altered by feature pruning; instead, the tuning process adjusts neighbour weights or distance metrics without modifying the number of predictors. Therefore, the traditional narrative of reducing complexity to curb variance does not apply directly. The observed accuracy gain suggests that the tuned model achieves a more favourable balance, likely reducing bias by better capturing local data structures, while variance does not increase appreciably since the feature set remains unchanged. This improvement validates the tuning effort, as the performance enhancement is realised without sacrificing the representational capacity afforded by the full fifty‑eight dimensions.

Comparing the two K‑Nearest Neighbours versions, the tuned classifier distinctly outperforms its untuned predecessor, justifying the additional computational expense of parameter search. When placed alongside other models in the experimental suite, the tuned K‑Nearest Neighbours emerges as the joint strongest contender, with an accuracy of 0.9153, marginally exceeding the tuned logistic regression at 0.9131 and significantly surpassing the optimised logistic regression at 0.8632, which operates on only twelve features. The baseline K‑Nearest Neighbours, however, lags behind both tuned logistic regression and the top K‑Nearest Neighbours version, underscoring that while feature consistency preserves structural parity, only the refined hyperparameter configuration unlocks the model’s full predictive potential. Thus, the tuned K‑Nearest Neighbours classifier stands as the superior representative of its family and a competitive benchmark within the broader comparison.

### Random Forest Classifier

In [741]:
RF_MODEL_NAME = "Random Forest"

#### Random Forest Pipeline

In [742]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

In [743]:
rf = RandomForestClassifier(
	n_estimators=500,
	max_depth=20, 
	min_samples_split=5,
	min_samples_leaf=2, 
	max_features="sqrt",
	bootstrap=True, 
	n_jobs=-1, 
	random_state=RANDOM_STATE
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
random_forest_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("rf_classifier", rf)
])

random_forest_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('rf_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [744]:
y_pred = random_forest_pipeline.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{RF_MODEL_NAME} Classifier"] = {
    "accuracy": rf_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{RF_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.933
Precision (weighted):    0.943
Recall (weighted):       0.933
F1-Score (weighted):     0.932

Confusion Matrix:
-----------------
True Negatives:  514
False Positives: 21
False Negatives: 41
True Positives:  345

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94       535
           1       0.94      0.89      0.92       386

    accuracy                           0.93       921
   macro avg       0.93      0.93      0.93       921
weighted avg       0.93      0.93      0.93       921



In [745]:
pprint.pprint(classifier_metrics)

{'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
                              'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
                                          'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
                                          'no_features': 58},
 'Tuned and Optimized Logistic Regression Classifier': {'accuracy': 0.8631921824104235,
                                                        'no_features': 12}}


#### Search for the Best Combination of Hyperparameters for Random Forest Classifier

In [746]:
param_dist = {  
	"rf_classifier__n_estimators": [100, 200, 500],  
	"rf_classifier__max_depth": [None, 10, 20, 30],  
	"rf_classifier__min_samples_split": [2, 5, 10],  
	"rf_classifier__min_samples_leaf": [1, 2, 4],  
	"rf_classifier__max_features": ["sqrt", "log2"]  
}  
  
random_random_forest = RandomizedSearchCV(  
	random_forest_pipeline,  
	param_distributions=param_dist,  
	n_iter=20,  
	cv=5,  
	scoring="accuracy",  
	n_jobs=-1,  
	random_state=42  
)  
  
random_random_forest.fit(X_train, y_train)  

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._state=777))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'rf_classifier__max_depth': [None, 10, ...], 'rf_classifier__max_features': ['sqrt', 'log2'], 'rf_classifier__min_samples_leaf': [1, 2, ...], 'rf_classifier__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide `

In [747]:
best_random_forest = random_random_forest.best_estimator_  
classifer = best_random_forest.named_steps["rf_classifier"]

print("Best Parameters:", random_random_forest.best_params_)
print("Best CV Score:", random_random_forest.best_score_)

y_pred = best_random_forest.predict(X_test)
tuned_rf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name="Tuned Random Forest Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'rf_classifier__n_estimators': 200, 'rf_classifier__min_samples_split': 2, 'rf_classifier__min_samples_leaf': 1, 'rf_classifier__max_features': 'log2', 'rf_classifier__max_depth': 30}
Best CV Score: 0.9538043478260869
TUNED RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.946
Precision (weighted):    0.952
Recall (weighted):       0.946
F1-Score (weighted):     0.946

Confusion Matrix:
-----------------
True Negatives:  517
False Positives: 18
False Negatives: 32
True Positives:  354

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95       535
           1       0.95      0.92      0.93       386

    accuracy                           0.95       921
   macro avg       0.95      0.94      0.94       921
weighted avg       0.95      0.95      0.95       921



In [748]:
classifier_metrics[f"Tuned {RF_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_rf_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
                              'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
                                          'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
                                          'no_features': 58},
 'Tuned Random Forest Classifier': {'accuracy': 0.9457111834961998,
                                    'no_features': 58},
 'Tuned and Optimized Logistic Regression Classifier': {'accuracy': 0.8631921824104235,
                                                        'no_features': 12}}


#### Optimizing Random Forest

In [749]:
# Best Parameters: 
# {'rf_classifier__n_estimators': 200, 
# 'rf_classifier__min_samples_split': 2, 
# 'rf_classifier__min_samples_leaf': 1, 
# 'rf_classifier__max_features': 'log2', 
# 'rf_classifier__max_depth': 30}


In [ ]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

char_freq_%21                 0.107670
capital_run_length_average    0.075169
char_freq_%24                 0.071830
word_freq_free                0.064465
word_freq_remove              0.060242
capital_run_length_longest    0.054236
word_freq_your                0.051724
capital_run_length_total      0.050939
word_freq_hp                  0.042302
word_freq_money               0.038980
word_freq_you                 0.034430
word_freq_our                 0.032908
word_freq_000                 0.026559
word_freq_hpl                 0.020101
word_freq_george              0.017617
word_freq_edu                 0.015870
word_freq_1999                0.014950
word_freq_receive             0.013988
word_freq_internet            0.013340
char_freq_%28                 0.013049
word_freq_over                0.012236
word_freq_will                0.011734
word_freq_all                 0.011664
word_freq_mail                0.011597
word_freq_email               0.010547
word_freq_re             

In [751]:
important_features_scores = feature_importance[feature_importance > 0.03]
important_features = feature_importance[feature_importance > 0.03].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

char_freq_%21                 0.107670
capital_run_length_average    0.075169
char_freq_%24                 0.071830
word_freq_free                0.064465
word_freq_remove              0.060242
capital_run_length_longest    0.054236
word_freq_your                0.051724
capital_run_length_total      0.050939
word_freq_hp                  0.042302
word_freq_money               0.038980
word_freq_you                 0.034430
word_freq_our                 0.032908
dtype: float64
NUMBER OF FEATURES: 12


In [752]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

In [753]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='log2',
    random_state=RANDOM_STATE,
    n_jobs=-1,  # Use all available cores
    verbose=1  # Show progress during training
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
random_forest_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("rf_classifier", rf)
])

random_forest_pipeline.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:    0.2s finished


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('rf_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [754]:
y_pred = random_forest_pipeline.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {RF_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.931
Precision (weighted):    0.935
Recall (weighted):       0.931
F1-Score (weighted):     0.930

Confusion Matrix:
-----------------
True Negatives:  511
False Positives: 24
False Negatives: 40
True Positives:  346

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94       535
           1       0.94      0.90      0.92       386

    accuracy                           0.93       921
   macro avg       0.93      0.93      0.93       921
weighted avg       0.93      0.93      0.93       921



[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished


In [755]:
classifier_metrics[f"Tuned and Optimized {RF_MODEL_NAME} Classifier"] = {
    "accuracy": rf_accuracy,
    "no_features": len(important_features_scores)
}

In [756]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Tuned Random Forest Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Random Forest Classifier',
  {'accuracy': 0.9326818675352877, 'no_features': 58}),
 ('Tuned and Optimized Random Forest Classifier',
  {'accuracy': 0.9305103148751357, 'no_features': 12}),
 ('Tuned K-Nearest Neighbors Classifier',
  {'accuracy': 0.9153094462540716, 'no_features': 58}),
 ('Logistic Regression Classifier',
  {'accuracy': 0.9142236699239956, 'no_features': 58}),
 ('Tuned Logistic Regression Classifier',
  {'accuracy': 0.9131378935939196, 'no_features': 58}),
 ('K-Nearest Neighbors Classifier',
  {'accuracy': 0.8935939196525515, 'no_features': 58}),
 ('Tuned and Optimized Logistic Regression Classifier',
  {'accuracy': 0.8631921824104235, 'no_features': 12})]


#### Endnotes

The random forest classifier underwent a three-stage evolution, beginning with a baseline model that utilised fifty-eight features and achieved an accuracy of 0.9327. The first refinement, a tuned version, retained the same number of features but elevated performance to 0.9457, representing a notable improvement through hyperparameter optimisation alone. The final iteration, a tuned and optimised variant, involved a substantial reduction in the feature space, contracting it to only twelve predictors. This compression, however, precipitated a decline in accuracy to 0.9305, a value slightly below the original baseline and markedly inferior to the tuned counterpart.

This progression offers a compelling illustration of the bias-variance tradeoff. The tuned model, operating on the full fifty-eight features, achieves the highest accuracy, suggesting that its ensemble of trees benefits from the rich informational content of the complete set without succumbing to overfitting. Conversely, the optimised model, by aggressively pruning to twelve features, intentionally increases bias in pursuit of lower variance and greater generalisability. Yet the empirical outcome is unequivocal: the reduction in complexity was not beneficial, as the accuracy drop of 1.5 percentage points from the tuned version indicates that the discarded features carried critical predictive signals that could not be compensated by the remaining dozen.

When comparing the best-performing random forest version—the tuned classifier at 0.9457—with the other models in the experimental suite, its supremacy is clear. It outperforms the tuned K-nearest neighbours at 0.9153, the logistic regression variants at 0.9142 and 0.9131, and the baseline K-nearest neighbours at 0.8936. Even the optimised random forest, despite its reduced complexity, still surpasses all non-random forest models except the tuned random forest itself, achieving 0.9305 against the best K-nearest neighbours at 0.9153. Nevertheless, within the random forest family, the tuned version stands unchallenged, demonstrating that for this ensemble method, preserving feature richness while fine-tuning hyperparameters yields the most robust predictive performance. The reduction in complexity, therefore, was not justified, as it sacrificed accuracy without delivering any competitive advantage over the simpler alternatives.

### Extreme Random Forest

In [757]:
XRF_MODEL_NAME = "Extreme Random Forest"

In [758]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Extreme Random Forest Pipeline

In [ ]:
xrf = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=False,
    criterion="gini",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()) 
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
extreme_random_forest_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("xrf_classifier", xrf)
])

extreme_random_forest_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('xrf_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [760]:
y_pred = extreme_random_forest_pipeline.predict(X_test)
xrf_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{XRF_MODEL_NAME} Classifier"] = {
    "accuracy": xrf_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{XRF_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

EXTREME RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.955
Precision (weighted):    0.960
Recall (weighted):       0.955
F1-Score (weighted):     0.955

Confusion Matrix:
-----------------
True Negatives:  520
False Positives: 15
False Negatives: 26
True Positives:  360

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       535
           1       0.96      0.93      0.95       386

    accuracy                           0.96       921
   macro avg       0.96      0.95      0.95       921
weighted avg       0.96      0.96      0.96       921



#### Random Search of Hyperparameters for Extreme Random Forest

In [ ]:
# extended hyperparameter tuning space
param_dist = {
    "xrf_classifier__n_estimators": [100, 200, 300, 500, 800],
    "xrf_classifier__criterion": ["gini", "entropy", "log_loss"],
    "xrf_classifier__max_depth": [None, 10, 20, 30, 50],
    "xrf_classifier__min_samples_split": [2, 5, 10, 20],
    "xrf_classifier__min_samples_leaf": [1, 2, 4, 8],
    "xrf_classifier__max_features": ["sqrt", "log2", None, 0.5, 0.8],
    "xrf_classifier__bootstrap": [True],
    "xrf_classifier__max_samples": [None, 0.5, 0.7, 0.9],
    "xrf_classifier__min_impurity_decrease": [0.0, 0.01, 0.05],
    "xrf_classifier__ccp_alpha": [0.0, 0.001, 0.01],
    "xrf_classifier__class_weight": [None, "balanced", "balanced_subsample"]
}

search = RandomizedSearchCV(
    extreme_random_forest_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)

search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._state=777))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'xrf_classifier__bootstrap': [True], 'xrf_classifier__ccp_alpha': [0.0, 0.001, ...], 'xrf_classifier__class_weight': [None, 'balanced', ...], 'xrf_classifier__criterion': ['gini', 'entropy', ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User G

In [762]:
best_extreme_random_forest = search.best_estimator_  
classifer = best_extreme_random_forest.named_steps["xrf_classifier"]

print("Best Parameters:", search.best_params_)
print("Best CV Score:", search.best_score_)

y_pred = best_extreme_random_forest.predict(X_test)
tuned_xrf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {XRF_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'xrf_classifier__n_estimators': 200, 'xrf_classifier__min_samples_split': 2, 'xrf_classifier__min_samples_leaf': 2, 'xrf_classifier__min_impurity_decrease': 0.0, 'xrf_classifier__max_samples': 0.5, 'xrf_classifier__max_features': 0.5, 'xrf_classifier__max_depth': 20, 'xrf_classifier__criterion': 'log_loss', 'xrf_classifier__class_weight': None, 'xrf_classifier__ccp_alpha': 0.0, 'xrf_classifier__bootstrap': True}
Best CV Score: 0.9426630434782609
TUNED EXTREME RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.934
Precision (weighted):    0.940
Recall (weighted):       0.934
F1-Score (weighted):     0.934

Confusion Matrix:
-----------------
True Negatives:  513
False Positives: 22
False Negatives: 39
True Positives:  347

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94       535
           1       0.94      0.90      0.92       386

    accuracy                           0.9

In [763]:
classifier_metrics[f"Tuned {XRF_MODEL_NAME}"] = {
    "accuracy": tuned_xrf_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
                              'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
                                 'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
                                          'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
                                          'no_features': 58},
 'Tuned Random Forest Classifier': {'accuracy': 0.9457111834961998,
                                    'no_features': 58}

#### Optimizing Extreme Random Forest

In [764]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

char_freq_%24                 0.078482
word_freq_remove              0.078191
word_freq_your                0.072690
char_freq_%21                 0.070127
word_freq_hp                  0.062518
capital_run_length_longest    0.053313
word_freq_free                0.048878
word_freq_000                 0.047398
capital_run_length_average    0.036053
word_freq_george              0.033069
word_freq_our                 0.032912
word_freq_money               0.029924
word_freq_you                 0.027134
word_freq_hpl                 0.026575
capital_run_length_total      0.024649
word_freq_edu                 0.017042
word_freq_receive             0.016872
word_freq_business            0.016543
word_freq_1999                0.016066
word_freq_over                0.015618
word_freq_internet            0.015264
word_freq_all                 0.014374
word_freq_re                  0.013286
word_freq_will                0.012943
word_freq_order               0.011452
word_freq_email          

In [765]:
important_features_scores = feature_importance[feature_importance > 0.028]
important_features = feature_importance[feature_importance > 0.028].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

char_freq_%24                 0.078482
word_freq_remove              0.078191
word_freq_your                0.072690
char_freq_%21                 0.070127
word_freq_hp                  0.062518
capital_run_length_longest    0.053313
word_freq_free                0.048878
word_freq_000                 0.047398
capital_run_length_average    0.036053
word_freq_george              0.033069
word_freq_our                 0.032912
word_freq_money               0.029924
dtype: float64
NUMBER OF FEATURES: 12


In [766]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

In [767]:
# Best Parameters: 
# {'xrf_classifier__n_estimators': 200, 
# 'xrf_classifier__min_samples_split': 2, 
# 'xrf_classifier__min_samples_leaf': 2, 
# 'xrf_classifier__min_impurity_decrease': 0.0, 
# 'xrf_classifier__max_samples': 0.5, 
# 'xrf_classifier__max_features': 0.5, 
# 'xrf_classifier__max_depth': 20, 
# 'xrf_classifier__criterion': 'log_loss', 
# 'xrf_classifier__class_weight': None, 
# 'xrf_classifier__ccp_alpha': 0.0, 
# 'xrf_classifier__bootstrap': True}

In [768]:
xrf = ExtraTreesClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=2,
    min_samples_leaf=2,
    min_impurity_decrease=0.0,
    max_features=0.5,
    max_samples=0.5,
    criterion='log_loss',
    class_weight=None,
    ccp_alpha=0.0,
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,  
    verbose=1  
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
extreme_random_forest_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("xrf_classifier", xrf)
])

extreme_random_forest_pipeline.fit(X_train, y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:    0.2s finished


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('xrf_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [769]:
y_pred = extreme_random_forest_pipeline.predict(X_test)
xrf_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {XRF_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED EXTREME RANDOM FOREST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.927
Precision (weighted):    0.942
Recall (weighted):       0.927
F1-Score (weighted):     0.927

Confusion Matrix:
-----------------
True Negatives:  514
False Positives: 21
False Negatives: 46
True Positives:  340

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.96      0.94       535
           1       0.94      0.88      0.91       386

    accuracy                           0.93       921
   macro avg       0.93      0.92      0.92       921
weighted avg       0.93      0.93      0.93       921



[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 200 out of 200 | elapsed:    0.0s finished


In [770]:
classifier_metrics[f"Tuned and Optimized {XRF_MODEL_NAME} Classifier"] = {
    "accuracy": xrf_accuracy,
    "no_features": len(important_features_scores)
}

In [771]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Random Forest Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Extreme Random Forest',
  {'accuracy': 0.9337676438653637, 'no_features': 58}),
 ('Random Forest Classifier',
  {'accuracy': 0.9326818675352877, 'no_features': 58}),
 ('Tuned and Optimized Random Forest Classifier',
  {'accuracy': 0.9305103148751357, 'no_features': 12}),
 ('Tuned and Optimized Extreme Random Forest Classifier',
  {'accuracy': 0.9272529858849077, 'no_features': 12}),
 ('Tuned K-Nearest Neighbors Classifier',
  {'accuracy': 0.9153094462540716, 'no_features': 58}),
 ('Logistic Regression Classifier',
  {'accuracy': 0.9142236699239956, 'no_features': 58}),
 ('Tuned Logistic Regression Classifier',
  {'accuracy': 0.9131378935939196, 'no_features': 58}),
 ('K-Nearest Neighbors Classifier',
  {'accuracy': 0.8935939196525515, 'no_features': 58}),
 ('Tuned and Optimized Logistic Regress

#### Endnotes

The extreme random forest classifier was examined across three distinct versions, each reflecting a different strategic approach to model construction. The initial extreme random forest model, operating on fifty-eight features, achieved a remarkable accuracy of 0.9555, establishing a very high baseline for subsequent iterations. A tuned version of this classifier, however, retained the same feature count but witnessed a decline in performance to 0.9338, a drop of over two percentage points, suggesting that the tuning process may have inadvertently compromised the model's inherent strengths. The final iteration, a tuned and optimised version, underwent a substantial reduction in the feature space to only twelve predictors, resulting in a further accuracy decrease to 0.9273.

The bias-variance tradeoff is clearly illustrated through this progression. The original extreme random forest, with its full complement of fifty-eight features, achieves the highest accuracy, indicating that its inherent randomness and ensemble structure effectively manage variance while capturing complex patterns. The tuned version, despite maintaining the same feature dimensionality, introduces hyperparameter adjustments that appear to increase bias without a corresponding variance reduction, as evidenced by the performance degradation. The optimised version, with its aggressive feature pruning to twelve predictors, deliberately increases bias to lower variance, yet the outcome is unequivocally detrimental; the accuracy falls to its lowest point, demonstrating that the discarded features contained essential predictive information that could not be compensated by the remaining subset.

Comparing the best-performing version, the original extreme random forest at 0.9555, with the other models in the experimental suite reveals its clear superiority. It outperforms the tuned random forest at 0.9457, the random forest baseline at 0.9327, and the tuned extreme random forest at 0.9338. Even when placed alongside non-ensemble models, the best extreme random forest maintains a significant margin, surpassing the tuned K-nearest neighbours at 0.9153 and all logistic regression variants, whose accuracies range from 0.9142 down to 0.8632. Within the extreme random forest family itself, the original version stands unchallenged, as both subsequent modifications failed to improve upon its performance. The reduction in complexity, whether through tuning or feature optimisation, was therefore not justified, as each step away from the original configuration resulted in a measurable loss of predictive accuracy.

### Multilayer Perceptron

In [772]:
MLP_MODEL_NAME = "Multilayer Perceptor"

In [773]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Multilayer Perceptron Pipeline

In [774]:
mlp = MLPClassifier(  
	hidden_layer_sizes=(64, 32),  
	activation="relu",  
	solver="adam",  
	max_iter=300,  
	random_state=RANDOM_STATE  
) 

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
mlp_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("mlp_classifier", mlp)
])

mlp_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('mlp_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [775]:
y_pred = mlp_pipeline.predict(X_test)
mlp_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{MLP_MODEL_NAME} Classifier"] = {
    "accuracy": mlp_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{MLP_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

MULTILAYER PERCEPTOR CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.924
Precision (weighted):    0.932
Recall (weighted):       0.924
F1-Score (weighted):     0.924

Confusion Matrix:
-----------------
True Negatives:  510
False Positives: 25
False Negatives: 45
True Positives:  341

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.95      0.94       535
           1       0.93      0.88      0.91       386

    accuracy                           0.92       921
   macro avg       0.93      0.92      0.92       921
weighted avg       0.92      0.92      0.92       921



In [776]:
pprint.pprint(classifier_metrics)

{'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
                              'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
                                 'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
                                          'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
                                          'no_features': 5

#### Random Search of Hyperparameters for Multiplayer Perceptor

In [777]:
param_dist = {  
	"mlp_classifier__hidden_layer_sizes": [(50,), (100,), (100, 50), (128, 64), (64, 32, 16)],  
	"mlp_classifier__activation": ["relu", "tanh"],  
	"mlp_classifier__solver": ["adam", "sgd"],  
	"mlp_classifier__alpha": [1e-5, 1e-4, 1e-3, 1e-2],  
	"mlp_classifier__learning_rate": ["constant", "adaptive"],  
	"mlp_classifier__learning_rate_init": [0.001, 0.01, 0.1],  
	"mlp_classifier__batch_size": [32, 64, 128],  
	"mlp_classifier__max_iter": [200, 300, 500]  
}  
  
random_search_mlp = RandomizedSearchCV(  
	mlp_pipeline,  
	param_distributions=param_dist,  
	n_iter=20,  
	cv=5,  
	scoring="accuracy",  
	n_jobs=-1,  
	verbose=2,  
	random_state=RANDOM_STATE  
)  
  
random_search_mlp.fit(X_train, y_train)  

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._state=777))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'mlp_classifier__activation': ['relu', 'tanh'], 'mlp_classifier__alpha': [1e-05, 0.0001, ...], 'mlp_classifier__batch_size': [32, 64, ...], 'mlp_classifier__hidden_layer_sizes': [(50,), (100,), ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`Use

In [778]:
best_mlp = random_search_mlp.best_estimator_  
classifer = best_mlp.named_steps["mlp_classifier"]

print("Best Parameters:", random_search_mlp.best_params_)
print("Best CV Score:", random_search_mlp.best_score_)

y_pred = best_mlp.predict(X_test)
tuned_mlp_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {MLP_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'mlp_classifier__solver': 'adam', 'mlp_classifier__max_iter': 200, 'mlp_classifier__learning_rate_init': 0.001, 'mlp_classifier__learning_rate': 'constant', 'mlp_classifier__hidden_layer_sizes': (100, 50), 'mlp_classifier__batch_size': 128, 'mlp_classifier__alpha': 0.01, 'mlp_classifier__activation': 'tanh'}
Best CV Score: 0.9372282608695652
TUNED MULTILAYER PERCEPTOR CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.933
Precision (weighted):    0.911
Recall (weighted):       0.933
F1-Score (weighted):     0.933

Confusion Matrix:
-----------------
True Negatives:  500
False Positives: 35
False Negatives: 27
True Positives:  359

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       535
           1       0.91      0.93      0.92       386

    accuracy                           0.93       921
   macro avg       0.93      0.93      0.93       921
weighted avg       0.93      0.93      0.

In [779]:
classifier_metrics[f"Tuned {MLP_MODEL_NAME}"] = {
    "accuracy": tuned_mlp_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
                              'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
                                 'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
                                          'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
                                          'no_features': 5

### Gradient Boosting Classifier

In [780]:
GB_MODEL_NAME = "Gradient Boosting"

In [781]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Gradient Boosting Base Pipeline

In [782]:
gradient_boosting = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
gb_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("gb_classifier", gradient_boosting)
])

gb_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('gb_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [783]:
y_pred = gb_pipeline.predict(X_test)
gb_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{GB_MODEL_NAME} Classifier"] = {
    "accuracy": gb_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{GB_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.939
Precision (weighted):    0.941
Recall (weighted):       0.939
F1-Score (weighted):     0.939

Confusion Matrix:
-----------------
True Negatives:  513
False Positives: 22
False Negatives: 34
True Positives:  352

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95       535
           1       0.94      0.91      0.93       386

    accuracy                           0.94       921
   macro avg       0.94      0.94      0.94       921
weighted avg       0.94      0.94      0.94       921



In [784]:
classifier_metrics

{'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
  'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
  'no_features': 58},
 'Tuned and Optimized Logistic Regression Classifier': {'accuracy': 0.8631921824104235,
  'no_features': 12},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
  'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
  'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Tuned Random Forest Classifier': {'accuracy': 0.9457111834961998,
  'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9305103148751357,
  'no_features': 12},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
  'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
  'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'ac

#### Search for the Best Combination of Hyperparameters for Gradient Boosting

In [785]:
param_dist = {
    "gb_classifier__n_estimators": [100, 200, 300],
    "gb_classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gb_classifier__max_depth": [3, 4, 5],
    "gb_classifier__min_samples_split": [2, 5, 10],
    "gb_classifier__min_samples_leaf": [1, 2, 4],
    "gb_classifier__subsample": [0.6, 0.8, 1.0],
    "gb_classifier__max_features": ["sqrt", "log2", None]
}

random_gb = RandomizedSearchCV(
    gb_pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_gb.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...sample=0.8))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'gb_classifier__learning_rate': [0.01, 0.05, ...], 'gb_classifier__max_depth': [3, 4, ...], 'gb_classifier__max_features': ['sqrt', 'log2', ...], 'gb_classifier__min_samples_leaf': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guid

In [786]:
best_gb = random_gb.best_estimator_  
classifer = best_gb.named_steps["gb_classifier"]

print("Best Parameters:", random_gb.best_params_)
print("Best CV Score:", random_gb.best_score_)

y_pred = best_gb.predict(X_test)
tuned_gb_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {GB_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'gb_classifier__subsample': 1.0, 'gb_classifier__n_estimators': 200, 'gb_classifier__min_samples_split': 10, 'gb_classifier__min_samples_leaf': 1, 'gb_classifier__max_features': 'log2', 'gb_classifier__max_depth': 5, 'gb_classifier__learning_rate': 0.05}
Best CV Score: 0.954891304347826
TUNED GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.946
Precision (weighted):    0.944
Recall (weighted):       0.946
F1-Score (weighted):     0.946

Confusion Matrix:
-----------------
True Negatives:  514
False Positives: 21
False Negatives: 29
True Positives:  357

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.95       535
           1       0.94      0.92      0.93       386

    accuracy                           0.95       921
   macro avg       0.95      0.94      0.94       921
weighted avg       0.95      0.95      0.95       921



In [787]:
classifier_metrics[f"Tuned {GB_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_gb_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
                                  'no_features': 58},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
                              'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
                                 'no_features': 58},
 'Tuned Gradient Boosting Classifier': {'accuracy': 0.9457111834961998,
                                        'no_features': 58},
 'Tuned K-Neares

#### Optimizing Gradient Boosting

In [788]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

char_freq_%24                 0.112690
char_freq_%21                 0.099300
word_freq_hp                  0.091311
word_freq_remove              0.071437
capital_run_length_longest    0.061488
capital_run_length_average    0.059587
word_freq_free                0.057158
word_freq_your                0.048076
capital_run_length_total      0.036953
word_freq_george              0.032641
word_freq_money               0.031458
word_freq_hpl                 0.029770
word_freq_edu                 0.024488
word_freq_000                 0.023086
word_freq_our                 0.022736
word_freq_receive             0.022062
word_freq_business            0.019130
word_freq_internet            0.012659
word_freq_mail                0.011809
word_freq_1999                0.010412
word_freq_you                 0.010175
word_freq_meeting             0.008325
word_freq_all                 0.007746
word_freq_re                  0.007298
word_freq_address             0.007068
word_freq_over           

In [789]:
important_features_scores = feature_importance[feature_importance > 0.029]
important_features = feature_importance[feature_importance > 0.029].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

char_freq_%24                 0.112690
char_freq_%21                 0.099300
word_freq_hp                  0.091311
word_freq_remove              0.071437
capital_run_length_longest    0.061488
capital_run_length_average    0.059587
word_freq_free                0.057158
word_freq_your                0.048076
capital_run_length_total      0.036953
word_freq_george              0.032641
word_freq_money               0.031458
word_freq_hpl                 0.029770
dtype: float64
NUMBER OF FEATURES: 12


In [790]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

In [791]:
# Best Parameters: 
# {'gb_classifier__subsample': 1.0, 
# 'gb_classifier__n_estimators': 200, 
# 'gb_classifier__min_samples_split': 10, 
# 'gb_classifier__min_samples_leaf': 1, 
# 'gb_classifier__max_features': 'log2', 
# 'gb_classifier__max_depth': 5, 
# 'gb_classifier__learning_rate': 0.05}


In [792]:
gradient_boosting = GradientBoostingClassifier(
    subsample=1.0,
    n_estimators=200,
    min_samples_split=10,
    min_samples_leaf=1,
    max_features="log2",
    max_depth=5,
    learning_rate=0.05
)

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
gb_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("gb_classifier", gradient_boosting)
])

gb_pipeline.fit(X_train_optimized, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('gb_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transf

In [793]:
y_pred = gb_pipeline.predict(X_test)
gb_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {GB_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.919
Precision (weighted):    0.926
Recall (weighted):       0.919
F1-Score (weighted):     0.918

Confusion Matrix:
-----------------
True Negatives:  508
False Positives: 27
False Negatives: 48
True Positives:  338

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       535
           1       0.93      0.88      0.90       386

    accuracy                           0.92       921
   macro avg       0.92      0.91      0.92       921
weighted avg       0.92      0.92      0.92       921



In [794]:
classifier_metrics[f"Tuned and Optimized {GB_MODEL_NAME} Classifier"] = {
    "accuracy": gb_accuracy,
    "no_features": len(important_features_scores)
}

In [795]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Random Forest Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Gradient Boosting Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Gradient Boosting Classifier',
  {'accuracy': 0.9391965255157437, 'no_features': 58}),
 ('Tuned Extreme Random Forest',
  {'accuracy': 0.9337676438653637, 'no_features': 58}),
 ('Random Forest Classifier',
  {'accuracy': 0.9326818675352877, 'no_features': 58}),
 ('Tuned Multilayer Perceptor',
  {'accuracy': 0.9326818675352877, 'no_features': 58}),
 ('Tuned and Optimized Random Forest Classifier',
  {'accuracy': 0.9305103148751357, 'no_features': 12}),
 ('Tuned and Optimized Extreme Random Forest Classifier',
  {'accuracy': 0.9272529858849077, 'no_features': 12}),
 ('Multilayer Perceptor Classifier',
  {'accuracy': 0.9239956568946797, 'no_features': 58}),
 ('Tuned and Optimized Gradient Boosting Classifier'

#### Endnotes

The gradient boosting model was developed through three successive versions, each characterised by distinct modifications to its configuration. The initial gradient boosting classifier, trained on fifty-eight features, achieved a baseline accuracy of 0.9392. A subsequent tuned version retained the full feature set of fifty-eight variables and improved performance to 0.9457, representing a meaningful gain through hyperparameter optimisation. The final iteration, a tuned and optimised gradient boosting classifier, underwent a dramatic reduction in the feature space to only twelve predictors, but this simplification came at a considerable cost, as its accuracy dropped sharply to 0.9186, falling below both previous versions and even below the original baseline.

This trajectory provides a clear demonstration of the bias-variance tradeoff in ensemble learning. The tuned gradient boosting model, operating on fifty-eight features, achieves its highest accuracy by leveraging the complete informational content of the dataset, suggesting that the boosting algorithm effectively manages variance through its sequential error-correction mechanism while keeping bias low. The optimised version, however, intentionally increases bias by pruning to twelve features in an effort to reduce variance and enhance generalisability. The empirical evidence, however, refutes the benefit of this strategy; the accuracy decline of 2.7 percentage points from the tuned version indicates that the removed features carried substantial predictive value that could not be recovered by the remaining dozen variables. The reduction in complexity was therefore not justified, as the simpler model failed to deliver competitive performance.

When comparing the best-performing gradient boosting version, the tuned classifier at 0.9457, against the broader experimental suite, it stands among the top performers. It matches the tuned random forest at the same accuracy and surpasses the extreme random forest baseline at 0.9555, though it falls short of that model's peak. The tuned gradient boosting comfortably outperforms the multilayer perceptron variants at 0.9327 and 0.9240, as well as all K-nearest neighbours and logistic regression classifiers, whose accuracies range from 0.9153 down to 0.8632. Within its own family, the tuned version is unequivocally superior to both the baseline and the optimised iteration. The optimised gradient boosting, despite its reduced complexity, still exceeds several simpler models but remains the weakest among the gradient boosting lineage, underscoring that for this powerful ensemble method, preserving feature richness while fine-tuning hyperparameters yields the most robust predictive performance.

### Extreme Gradient Boosting Classifier

In [796]:
from xgboost import XGBClassifier

XGB_MODEL_NAME = "Extreme Gradient Boosting"

In [797]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Extreme Gradient Boosting Base Pipeline

In [798]:
extreme_gradient_boosting = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric="logloss"
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
xgb_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("xgb_classifier", extreme_gradient_boosting)
])

xgb_pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('xgb_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [799]:
y_pred = xgb_pipeline.predict(X_test)
xgb_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{XGB_MODEL_NAME} Classifier"] = {
    "accuracy": xgb_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{XGB_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

EXTREME GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.948
Precision (weighted):    0.947
Recall (weighted):       0.948
F1-Score (weighted):     0.948

Confusion Matrix:
-----------------
True Negatives:  515
False Positives: 20
False Negatives: 28
True Positives:  358

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       535
           1       0.95      0.93      0.94       386

    accuracy                           0.95       921
   macro avg       0.95      0.95      0.95       921
weighted avg       0.95      0.95      0.95       921



In [800]:
classifier_metrics

{'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
  'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
  'no_features': 58},
 'Tuned and Optimized Logistic Regression Classifier': {'accuracy': 0.8631921824104235,
  'no_features': 12},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
  'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
  'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Tuned Random Forest Classifier': {'accuracy': 0.9457111834961998,
  'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9305103148751357,
  'no_features': 12},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
  'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
  'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'ac

#### Search for the Best Combination of Hyperparameters for Extreme Gradient Boosting

In [801]:
param_grid = {
    "xgb_classifier__n_estimators": [100, 200, 300],
    "xgb_classifier__max_depth": [3, 4, 5, 6],
    "xgb_classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "xgb_classifier__subsample": [0.6, 0.8, 1.0],
    "xgb_classifier__colsample_bytree": [0.6, 0.8, 1.0],
    "xgb_classifier__gamma": [0, 0.1, 0.3, 0.5],
    "xgb_classifier__reg_alpha": [0, 0.1, 1],
    "xgb_classifier__reg_lambda": [1, 1.5, 2]
}

random_xgb = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_xgb.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'xgb_classifier__colsample_bytree': [0.6, 0.8, ...], 'xgb_classifier__gamma': [0, 0.1, ...], 'xgb_classifier__learning_rate': [0.01, 0.05, ...], 'xgb_classifier__max_depth': [3, 4, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` f

In [802]:
best_xgb = random_xgb.best_estimator_  
classifer = best_xgb.named_steps["xgb_classifier"]

print("Best Parameters:", random_xgb.best_params_)
print("Best CV Score:", random_xgb.best_score_)

y_pred = best_xgb.predict(X_test)
tuned_xgb_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {XGB_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'xgb_classifier__subsample': 0.6, 'xgb_classifier__reg_lambda': 2, 'xgb_classifier__reg_alpha': 1, 'xgb_classifier__n_estimators': 100, 'xgb_classifier__max_depth': 6, 'xgb_classifier__learning_rate': 0.2, 'xgb_classifier__gamma': 0.5, 'xgb_classifier__colsample_bytree': 0.8}
Best CV Score: 0.9516304347826086
TUNED EXTREME GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.950
Precision (weighted):    0.947
Recall (weighted):       0.950
F1-Score (weighted):     0.950

Confusion Matrix:
-----------------
True Negatives:  515
False Positives: 20
False Negatives: 26
True Positives:  360

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       535
           1       0.95      0.93      0.94       386

    accuracy                           0.95       921
   macro avg       0.95      0.95      0.95       921
weighted avg       0.95      0.95      0.95       921



In [803]:
classifier_metrics[f"Tuned {XGB_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_xgb_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Extreme Gradient Boosting Classifier': {'accuracy': 0.9478827361563518,
                                          'no_features': 58},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
                                  'no_features': 58},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
                              'no_features': 58},
 'Tuned Extreme Gradient Boosting Classifier': {'accuracy': 0.9500542888165038,
                                                'n

#### Optimizing Extreme Gradient Boosting Classifier

In [804]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

char_freq_%24                 0.154013
word_freq_remove              0.116680
char_freq_%21                 0.084148
word_freq_hp                  0.060839
word_freq_george              0.035350
word_freq_money               0.035106
word_freq_free                0.031456
word_freq_edu                 0.025546
word_freq_hpl                 0.023053
capital_run_length_longest    0.021570
word_freq_your                0.021429
word_freq_1999                0.020977
word_freq_650                 0.020847
word_freq_our                 0.020608
word_freq_meeting             0.020099
word_freq_85                  0.019321
capital_run_length_average    0.017542
word_freq_business            0.016148
word_freq_project             0.014688
word_freq_re                  0.014315
word_freq_credit              0.014271
word_freq_pm                  0.013406
word_freq_000                 0.012935
word_freq_internet            0.012437
capital_run_length_total      0.010985
word_freq_over           

In [805]:
important_features_scores = feature_importance[feature_importance > 0.028]
important_features = feature_importance[feature_importance > 0.028].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

char_freq_%24       0.154013
word_freq_remove    0.116680
char_freq_%21       0.084148
word_freq_hp        0.060839
word_freq_george    0.035350
word_freq_money     0.035106
word_freq_free      0.031456
dtype: float32
NUMBER OF FEATURES: 7


In [806]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

In [807]:
# Best Parameters: 
# {'xgb_classifier__subsample': 0.6,
# 'xgb_classifier__reg_lambda': 2, 
# 'xgb_classifier__reg_alpha': 1, 
# 'xgb_classifier__n_estimators': 100, 
# 'xgb_classifier__max_depth': 6, 
# 'xgb_classifier__learning_rate': 0.2, 
# 'xgb_classifier__gamma': 0.5, 
# 'xgb_classifier__colsample_bytree': 0.8}


In [808]:
extreme_gradient_boosting = XGBClassifier(
    n_estimators=100,
    learning_rate=0.2,
    max_depth=6,
    subsample=0.6,
    colsample_bytree=0.8,
    reg_alpha=2.0,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    gamma=0.5
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median"))  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
xgb_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("xgb_classifier", extreme_gradient_boosting)
])

xgb_pipeline.fit(X_train_optimized, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('xgb_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different trans

In [809]:
y_pred = xgb_pipeline.predict(X_test)
xgb_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {XGB_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED EXTREME GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.897
Precision (weighted):    0.903
Recall (weighted):       0.897
F1-Score (weighted):     0.896

Confusion Matrix:
-----------------
True Negatives:  500
False Positives: 35
False Negatives: 60
True Positives:  326

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91       535
           1       0.90      0.84      0.87       386

    accuracy                           0.90       921
   macro avg       0.90      0.89      0.89       921
weighted avg       0.90      0.90      0.90       921



In [810]:
classifier_metrics[f"Tuned and Optimized {XGB_MODEL_NAME} Classifier"] = {
    "accuracy": xgb_accuracy,
    "no_features": len(important_features_scores)
}

In [811]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9478827361563518, 'no_features': 58}),
 ('Tuned Random Forest Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Gradient Boosting Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Gradient Boosting Classifier',
  {'accuracy': 0.9391965255157437, 'no_features': 58}),
 ('Tuned Extreme Random Forest',
  {'accuracy': 0.9337676438653637, 'no_features': 58}),
 ('Random Forest Classifier',
  {'accuracy': 0.9326818675352877, 'no_features': 58}),
 ('Tuned Multilayer Perceptor',
  {'accuracy': 0.9326818675352877, 'no_features': 58}),
 ('Tuned and Optimized Random Forest Classifier',
  {'accuracy': 0.9305103148751357, 'no_features': 12}),
 ('Tuned and Optimized Extreme Random Forest Classifier',

#### Endnotes

The extreme gradient boosting model was examined across three distinct versions, each reflecting a different strategic approach to model configuration. The initial extreme gradient boosting classifier, operating on fifty-eight features, achieved a strong baseline accuracy of 0.9479. A subsequent tuned version retained the full feature set of fifty-eight variables and improved performance to 0.9501, representing a modest but meaningful gain through hyperparameter optimisation. The final iteration, a tuned and optimised extreme gradient boosting classifier, underwent a substantial reduction in the feature space to only seven predictors, but this aggressive simplification came at a severe cost, as its accuracy plummeted to 0.8969, falling well below both previous versions and even below several simpler models in the experimental suite.

This progression offers a stark illustration of the bias-variance tradeoff in gradient boosting frameworks. The tuned extreme gradient boosting model, leveraging the complete set of fifty-eight features, achieves its highest accuracy by exploiting the full informational richness of the dataset, suggesting that the algorithm's regularisation parameters effectively control variance while maintaining low bias. The optimised version, however, deliberately increases bias by pruning to only seven features in an effort to reduce variance and enhance generalisability. The empirical outcome, however, unequivocally refutes the benefit of this strategy; the accuracy decline of over five percentage points from the tuned version indicates that the discarded features contained indispensable predictive signals that could not be compensated by the remaining handful of variables. The reduction in complexity was therefore not justified, as the simplified model failed to deliver competitive performance and instead produced the weakest result among all gradient boosting variants.

When comparing the best-performing extreme gradient boosting version, the tuned classifier at 0.9501, against the broader experimental suite, it emerges as the second-best model overall, surpassed only by the extreme random forest at 0.9555. The tuned extreme gradient boosting comfortably outperforms the tuned random forest at 0.9457, the tuned gradient boosting at the same value, and all multilayer perceptron and K-nearest neighbours variants. Within its own family, the tuned version is unequivocally superior to both the baseline and the heavily pruned iteration, with the latter even falling behind the baseline K-nearest neighbours at 0.8936. This hierarchy underscores that for extreme gradient boosting, preserving feature richness while fine-tuning hyperparameters yields the most robust predictive performance, and any aggressive reduction in complexity proves detrimental to the model's effectiveness.

### Light Gradient Boosting Classifier 

In [812]:
from lightgbm import LGBMClassifier

LGBM_MODEL_NAME = "Light Gradient Boosting"

In [813]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = []

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Light Gradient Boosting Classifier Base Pipeline

In [814]:
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE
)


num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
lgbm_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("lgbm_classifier", lgbm)
])

lgbm_pipeline.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 1427, number of negative: 2253
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000926 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6996
[LightGBM] [Info] Number of data points in the train set: 3680, number of used features: 57
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.387772 -> initscore=-0.456688
[LightGBM] [Info] Start training from score -0.456688


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('lgbm_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

In [815]:
y_pred = lgbm_pipeline.predict(X_test)
lgbm_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{LGBM_MODEL_NAME} Classifier"] = {
    "accuracy": lgbm_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{LGBM_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

LIGHT GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.950
Precision (weighted):    0.947
Recall (weighted):       0.950
F1-Score (weighted):     0.950

Confusion Matrix:
-----------------
True Negatives:  515
False Positives: 20
False Negatives: 26
True Positives:  360

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96       535
           1       0.95      0.93      0.94       386

    accuracy                           0.95       921
   macro avg       0.95      0.95      0.95       921
weighted avg       0.95      0.95      0.95       921



In [816]:
classifier_metrics

{'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
  'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
  'no_features': 58},
 'Tuned and Optimized Logistic Regression Classifier': {'accuracy': 0.8631921824104235,
  'no_features': 12},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
  'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
  'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Tuned Random Forest Classifier': {'accuracy': 0.9457111834961998,
  'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9305103148751357,
  'no_features': 12},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
  'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
  'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'ac

#### Search for the Best Combination of Hyperparameters for Light Gradient Boosting Classifier

In [817]:
param_grid = {
    "lgbm_classifier__n_estimators": [100, 200, 300],
    "lgbm_classifier__num_leaves": [31, 50, 70, 100],
    "lgbm_classifier__max_depth": [-1, 3, 5, 7],
    "lgbm_classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "lgbm_classifier__subsample": [0.6, 0.8, 1.0],
    "lgbm_classifier__colsample_bytree": [0.6, 0.8, 1.0],
    "lgbm_classifier__reg_alpha": [0, 0.1, 1],
    "lgbm_classifier__reg_lambda": [0, 0.1, 1],
    "lgbm_classifier__min_child_samples": [20, 50, 100]
}

random_lgbm = RandomizedSearchCV(
    lgbm_pipeline,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_lgbm.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 1427, number of negative: 2253
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001099 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6981
[LightGBM] [Info] Number of data points in the train set: 3680, number of used features: 56
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.387772 -> initscore=-0.456688
[LightGBM] [Info] Start training from score -0.456688
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...sample=0.8))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'lgbm_classifier__colsample_bytree': [0.6, 0.8, ...], 'lgbm_classifier__learning_rate': [0.01, 0.05, ...], 'lgbm_classifier__max_depth': [-1, 3, ...], 'lgbm_classifier__min_child_samples': [20, 50, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:

In [818]:
best_lgbm = random_lgbm.best_estimator_  
classifer = best_lgbm.named_steps["lgbm_classifier"]

print("Best Parameters:", random_lgbm.best_params_)
print("Best CV Score:", random_lgbm.best_score_)

y_pred = best_lgbm.predict(X_test)
tuned_lgbm_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {LGBM_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'lgbm_classifier__subsample': 1.0, 'lgbm_classifier__reg_lambda': 0, 'lgbm_classifier__reg_alpha': 0, 'lgbm_classifier__num_leaves': 100, 'lgbm_classifier__n_estimators': 300, 'lgbm_classifier__min_child_samples': 50, 'lgbm_classifier__max_depth': 5, 'lgbm_classifier__learning_rate': 0.1, 'lgbm_classifier__colsample_bytree': 0.8}
Best CV Score: 0.9567934782608696
TUNED LIGHT GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.950
Precision (weighted):    0.955
Recall (weighted):       0.950
F1-Score (weighted):     0.950

Confusion Matrix:
-----------------
True Negatives:  518
False Positives: 17
False Negatives: 29
True Positives:  357

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       535
           1       0.95      0.92      0.94       386

    accuracy                           0.95       921
   macro avg       0.95      0.95      0.95       921
weighted avg    

In [819]:
classifier_metrics[f"Tuned {LGBM_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_lgbm_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Extreme Gradient Boosting Classifier': {'accuracy': 0.9478827361563518,
                                          'no_features': 58},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
                                  'no_features': 58},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Light Gradient Boosting Classifier': {'accuracy': 0.9500542888165038,
                                        'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
                              'no_features': 58}

#### Optimizing LGBM

In [820]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

capital_run_length_total      306
word_freq_you                 288
capital_run_length_average    256
char_freq_%21                 254
capital_run_length_longest    206
word_freq_your                180
word_freq_will                158
word_freq_free                154
char_freq_%28                 143
word_freq_our                 100
char_freq_%24                  97
word_freq_remove               95
word_freq_all                  95
word_freq_re                   91
word_freq_hp                   89
word_freq_edu                  88
word_freq_mail                 85
word_freq_over                 60
word_freq_email                58
word_freq_business             55
word_freq_1999                 54
word_freq_george               52
char_freq_%3B                  50
word_freq_meeting              47
word_freq_internet             45
word_freq_000                  43
word_freq_money                43
word_freq_pm                   41
word_freq_receive              38
word_freq_proj

In [821]:
important_features_scores = feature_importance[feature_importance > 94]
important_features = feature_importance[feature_importance > 94].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

capital_run_length_total      306
word_freq_you                 288
capital_run_length_average    256
char_freq_%21                 254
capital_run_length_longest    206
word_freq_your                180
word_freq_will                158
word_freq_free                154
char_freq_%28                 143
word_freq_our                 100
char_freq_%24                  97
word_freq_remove               95
word_freq_all                  95
dtype: int32
NUMBER OF FEATURES: 13


In [822]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

In [823]:
# Best Parameters: 
# {'lgbm_classifier__subsample': 1.0, 
# 'lgbm_classifier__reg_lambda': 0, 
# 'lgbm_classifier__reg_alpha': 0, 
# 'lgbm_classifier__num_leaves': 100, 
# 'lgbm_classifier__n_estimators': 300, 
# 'lgbm_classifier__min_child_samples': 50, 
# 'lgbm_classifier__max_depth': 5, 
# 'lgbm_classifier__learning_rate': 0.1, 
# 'lgbm_classifier__colsample_bytree': 0.8}


In [824]:
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    num_leaves=100,
    subsample=1.0,
    colsample_bytree=0.8,
    reg_alpha=0,
    reg_lambda=0,
    min_child_samples=50,
    random_state=RANDOM_STATE,
    eval_metric="logloss"
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
lgbm_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("lgbm_classifier", lgbm)
])

lgbm_pipeline.fit(X_train, y_train)

[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Info] Number of positive: 1427, number of negative: 2253
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000349 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2870
[LightGBM] [Info] Number of data points in the train set: 3680, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.387772 -> initscore=-0.456688
[LightGBM] [Info] Start training from score -0.456688
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('lgbm_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

In [825]:
y_pred = lgbm_pipeline.predict(X_test)
lgbm_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {LGBM_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

[LightGBM] [Warning] Unknown parameter: eval_metric
TUNED AND OPTIMIZED LIGHT GRADIENT BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.916
Precision (weighted):    0.926
Recall (weighted):       0.916
F1-Score (weighted):     0.916

Confusion Matrix:
-----------------
True Negatives:  508
False Positives: 27
False Negatives: 50
True Positives:  336

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       535
           1       0.93      0.87      0.90       386

    accuracy                           0.92       921
   macro avg       0.92      0.91      0.91       921
weighted avg       0.92      0.92      0.92       921



In [826]:
classifier_metrics[f"Tuned and Optimized {LGBM_MODEL_NAME} Classifier"] = {
    "accuracy": lgbm_accuracy,
    "no_features": len(important_features_scores)
}

In [827]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Light Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Tuned Light Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9478827361563518, 'no_features': 58}),
 ('Tuned Random Forest Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Gradient Boosting Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Gradient Boosting Classifier',
  {'accuracy': 0.9391965255157437, 'no_features': 58}),
 ('Tuned Extreme Random Forest',
  {'accuracy': 0.9337676438653637, 'no_features': 58}),
 ('Random Forest Classifier',
  {'accuracy': 0.9326818675352877, 'no_features': 58}),
 ('Tuned Multilayer Perceptor',
  {'accuracy': 0.9326

#### Endnotes

The light gradient boosting model was examined across three distinct versions, each reflecting a different strategic approach to model configuration. The initial light gradient boosting classifier, operating on fifty-eight features, achieved a strong accuracy of 0.9501. A subsequent tuned version retained the full feature set of fifty-eight variables and achieved exactly the same accuracy of 0.9501, indicating that the tuning process, while presumably altering hyperparameters, did not produce any measurable improvement in predictive performance for this particular dataset. The final iteration, a tuned and optimised light gradient boosting classifier, underwent a reduction in the feature space to thirteen predictors, but this simplification came at a noticeable cost, as its accuracy declined to 0.9164, falling significantly below both previous versions.

This progression offers a clear illustration of the bias-variance tradeoff within the light gradient boosting framework. The initial and tuned models, both leveraging the complete set of fifty-eight features, achieve the highest accuracy by exploiting the full informational richness of the dataset, suggesting that the algorithm's inherent regularisation effectively controls variance while maintaining low bias. The optimised version, however, deliberately increases bias by pruning to thirteen features in an effort to reduce variance and enhance generalisability. The empirical outcome, however, refutes the benefit of this strategy; the accuracy decline of over three percentage points from the tuned version indicates that the discarded features contained essential predictive signals that could not be compensated by the remaining variables. The reduction in complexity was therefore not justified, as the simplified model failed to deliver competitive performance and instead produced a result inferior to several simpler classifiers.

When comparing the best-performing light gradient boosting versions, the initial and tuned classifiers at 0.9501, against the broader experimental suite, they emerge as joint second-best models overall, surpassed only by the extreme random forest at 0.9555 and tied with the tuned extreme gradient boosting at the same accuracy. These light gradient boosting variants comfortably outperform the extreme gradient boosting baseline at 0.9479, the tuned random forest at 0.9457, and all multilayer perceptron and K-nearest neighbours variants. Within its own family, both the initial and tuned versions are unequivocally superior to the optimised iteration, with the latter even falling behind the tuned K-nearest neighbours at 0.9153. This hierarchy underscores that for light gradient boosting, preserving feature richness while fine-tuning hyperparameters yields the most robust predictive performance, and any aggressive reduction in complexity proves detrimental to the model's effectiveness.

### CatBoost Classifier

In [828]:
from catboost import CatBoostClassifier

CATBOOST_MODEL_NAME = "CatBoost"

In [829]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### CatBoost Classifier Base Pipeline

In [830]:
catboost = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    random_state=42,
    verbose=False
)


num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
catboost_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("catboost_classifier", catboost)
])

catboost_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('catboost_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

In [831]:
y_pred = catboost_pipeline.predict(X_test)
catboost_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{CATBOOST_MODEL_NAME} Classifier"] = {
    "accuracy": catboost_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{CATBOOST_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

CATBOOST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.942
Precision (weighted):    0.932
Recall (weighted):       0.942
F1-Score (weighted):     0.942

Confusion Matrix:
-----------------
True Negatives:  509
False Positives: 26
False Negatives: 27
True Positives:  359

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95       535
           1       0.93      0.93      0.93       386

    accuracy                           0.94       921
   macro avg       0.94      0.94      0.94       921
weighted avg       0.94      0.94      0.94       921



In [832]:
classifier_metrics

{'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
  'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
  'no_features': 58},
 'Tuned and Optimized Logistic Regression Classifier': {'accuracy': 0.8631921824104235,
  'no_features': 12},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
  'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
  'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Tuned Random Forest Classifier': {'accuracy': 0.9457111834961998,
  'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9305103148751357,
  'no_features': 12},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
  'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
  'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'ac

#### Search for the Best Combination of Hyperparameters for CatBoost Classifier

In [833]:
param_grid = {
    "catboost_classifier__iterations": [100, 200, 300],  
    "catboost_classifier__depth": [3, 5, 7, 9],
    "catboost_classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "catboost_classifier__subsample": [0.6, 0.8, 1.0],
    "catboost_classifier__colsample_bylevel": [0.6, 0.8, 1.0],
    "catboost_classifier__l2_leaf_reg": [0, 0.1, 1],  
    "catboost_classifier__min_child_samples": [20, 50, 100],
}

random_catboost = RandomizedSearchCV(
    catboost_pipeline,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_catboost.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...bose=False))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'catboost_classifier__colsample_bylevel': [0.6, 0.8, ...], 'catboost_classifier__depth': [3, 5, ...], 'catboost_classifier__iterations': [100, 200, ...], 'catboost_classifier__l2_leaf_reg': [0, 0.1, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref

In [834]:
best_catboost = random_catboost.best_estimator_  
classifer = best_catboost.named_steps["catboost_classifier"]

print("Best Parameters:", random_catboost.best_params_)
print("Best CV Score:", random_catboost.best_score_)

y_pred = best_catboost.predict(X_test)
tuned_catboost_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {CATBOOST_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'catboost_classifier__subsample': 0.6, 'catboost_classifier__min_child_samples': 20, 'catboost_classifier__learning_rate': 0.2, 'catboost_classifier__l2_leaf_reg': 1, 'catboost_classifier__iterations': 200, 'catboost_classifier__depth': 7, 'catboost_classifier__colsample_bylevel': 0.8}
Best CV Score: 0.9592391304347826
TUNED CATBOOST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.946
Precision (weighted):    0.944
Recall (weighted):       0.946
F1-Score (weighted):     0.946

Confusion Matrix:
-----------------
True Negatives:  514
False Positives: 21
False Negatives: 29
True Positives:  357

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.96      0.95       535
           1       0.94      0.92      0.93       386

    accuracy                           0.95       921
   macro avg       0.95      0.94      0.94       921
weighted avg       0.95      0.95      0.95       921



In [835]:
classifier_metrics[f"Tuned {CATBOOST_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_catboost_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'CatBoost Classifier': {'accuracy': 0.9424538545059717, 'no_features': 58},
 'Extreme Gradient Boosting Classifier': {'accuracy': 0.9478827361563518,
                                          'no_features': 58},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
                                  'no_features': 58},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Light Gradient Boosting Classifier': {'accuracy': 0.9500542888165038,
                                        'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
                                     'no_features': 58},
 'Random Forest Classifier': {'ac

#### Optimizing CatBoost Classifier

In [836]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

word_freq_george              10.336616
word_freq_hp                   7.270365
capital_run_length_average     6.268333
capital_run_length_longest     5.934324
char_freq_%21                  5.532921
char_freq_%24                  4.427405
word_freq_remove               4.392892
capital_run_length_total       4.247730
word_freq_edu                  3.938186
word_freq_free                 3.164758
word_freq_you                  3.092677
word_freq_our                  3.052979
word_freq_your                 2.964509
word_freq_will                 2.758455
word_freq_meeting              2.640583
char_freq_%28                  2.545807
word_freq_re                   2.445261
word_freq_business             1.811086
word_freq_1999                 1.725523
word_freq_email                1.381197
word_freq_money                1.374051
word_freq_85                   1.264360
word_freq_000                  1.224947
word_freq_technology           1.187199
word_freq_mail                 0.945775


In [837]:
important_features_scores = feature_importance[feature_importance > 3]
important_features = feature_importance[feature_importance > 3].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

word_freq_george              10.336616
word_freq_hp                   7.270365
capital_run_length_average     6.268333
capital_run_length_longest     5.934324
char_freq_%21                  5.532921
char_freq_%24                  4.427405
word_freq_remove               4.392892
capital_run_length_total       4.247730
word_freq_edu                  3.938186
word_freq_free                 3.164758
word_freq_you                  3.092677
word_freq_our                  3.052979
dtype: float64
NUMBER OF FEATURES: 12


In [838]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

In [839]:
# Best Parameters: 
# {'catboost_classifier__subsample': 0.6, 
# 'catboost_classifier__min_child_samples': 20, 
# 'catboost_classifier__learning_rate': 0.2, 
# 'catboost_classifier__l2_leaf_reg': 1, 
# 'catboost_classifier__iterations': 200, 
# 'catboost_classifier__depth': 7, 
# 'catboost_classifier__colsample_bylevel': 0.8}

In [840]:
catboost = CatBoostClassifier(
    iterations=200,
    depth=7,
    learning_rate=0.2,
    subsample=0.6,
    colsample_bylevel=0.8,
    l2_leaf_reg=1,
    min_child_samples=20,
    random_seed=RANDOM_STATE,
    verbose=100,  
)

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
catboost_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("catboost_classifier", catboost)
])

catboost_pipeline.fit(X_train, y_train)

0:	learn: 0.4736099	total: 12.2ms	remaining: 2.42s
100:	learn: 0.0475837	total: 488ms	remaining: 479ms
199:	learn: 0.0188322	total: 979ms	remaining: 0us


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('catboost_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

In [841]:
y_pred = catboost_pipeline.predict(X_test)
catboost_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {CATBOOST_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED CATBOOST CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.936
Precision (weighted):    0.936
Recall (weighted):       0.936
F1-Score (weighted):     0.936

Confusion Matrix:
-----------------
True Negatives:  511
False Positives: 24
False Negatives: 35
True Positives:  351

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95       535
           1       0.94      0.91      0.92       386

    accuracy                           0.94       921
   macro avg       0.94      0.93      0.93       921
weighted avg       0.94      0.94      0.94       921



In [842]:
classifier_metrics[f"Tuned and Optimized {CATBOOST_MODEL_NAME} Classifier"] = {
    "accuracy": catboost_accuracy,
    "no_features": len(important_features_scores)
}

In [843]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Light Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Tuned Light Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9478827361563518, 'no_features': 58}),
 ('Tuned Random Forest Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Gradient Boosting Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned CatBoost Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('CatBoost Classifier', {'accuracy': 0.9424538545059717, 'no_features': 58}),
 ('Gradient Boosting Classifier',
  {'accuracy': 0.9391965255157437, 'no_features': 58}),
 ('Tuned and Optimized CatBoost Classifier',
  {'accuracy': 0.

#### Endnotes

The CatBoost model was developed through three successive versions, each characterised by distinct modifications to its configuration. The initial CatBoost classifier, trained on fifty-eight features, achieved a baseline accuracy of 0.9425. A subsequent tuned version retained the full feature set of fifty-eight variables and improved performance to 0.9457, representing a modest but meaningful gain through hyperparameter optimisation. The final iteration, a tuned and optimised CatBoost classifier, underwent a substantial reduction in the feature space to twelve predictors, but this simplification came at a noticeable cost, as its accuracy declined to 0.9359, falling below both previous versions and even below the original baseline.

This progression provides a clear demonstration of the bias-variance tradeoff within the CatBoost framework. The tuned CatBoost model, leveraging the complete set of fifty-eight features, achieves its highest accuracy by exploiting the full informational richness of the dataset, suggesting that the algorithm's inherent handling of categorical features and regularisation effectively controls variance while maintaining low bias. The optimised version, however, deliberately increases bias by pruning to twelve features in an effort to reduce variance and enhance generalisability. The empirical outcome, however, refutes the benefit of this strategy; the accuracy decline of nearly one percentage point from the tuned version indicates that the discarded features contained predictive value that could not be compensated by the remaining variables. The reduction in complexity was therefore not justified, as the simplified model failed to deliver competitive performance and instead produced a result inferior to several other ensemble methods.

When comparing the best-performing CatBoost version, the tuned classifier at 0.9457, against the broader experimental suite, it occupies a solid mid-tier position among the top performers. It ties with the tuned random forest and tuned gradient boosting at the same accuracy, while falling behind the extreme random forest at 0.9555 and the light gradient boosting variants at 0.9501. The tuned CatBoost comfortably outperforms the gradient boosting baseline at 0.9392, all multilayer perceptron variants, and all K-nearest neighbours and logistic regression classifiers, whose accuracies range from 0.9153 down to 0.8632. Within its own family, the tuned version is unequivocally superior to both the baseline and the optimised iteration, with the latter still exceeding several simpler models but remaining the weakest among the CatBoost lineage. This hierarchy underscores that for CatBoost, preserving feature richness while fine-tuning hyperparameters yields the most robust predictive performance, and any aggressive reduction in complexity proves detrimental to the model's effectiveness.

### Ada Boosting Classifier

In [844]:
ADABOOST_MODEL_NAME = "Ada Boosting"

In [845]:
numerical_features = df.drop(columns=["class"]).select_dtypes(include=np.number).columns
categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(  
	X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE  
)

#### Ada Boosting Classifier Base Pipeline 

In [846]:
base_estimator = DecisionTreeClassifier(  
	max_depth=2,  
	min_samples_split=5,  
	min_samples_leaf=2,  
	random_state=42  
)  

adaboost = AdaBoostClassifier(  
	estimator=base_estimator,  
	n_estimators=200,  
	learning_rate=0.05,  
	# algorithm="SAMME.R",  
	random_state=RANDOM_STATE
)  

num_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  
])  
  
cat_pipeline = Pipeline([  
	("imputer", SimpleImputer(strategy="most_frequent")),  
	("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
	("num", num_pipeline, numerical_features),  
	("cat", cat_pipeline, categorical_features)  
])  
  
adaboost_pipeline = Pipeline([  
	("preprocessing", preprocessor),  
	("adaboost_classifier", adaboost)
])

adaboost_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('adaboost_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

In [847]:
y_pred = adaboost_pipeline.predict(X_test)
adaboost_accuracy = accuracy_score(y_test, y_pred)
classifier_metrics[f"{ADABOOST_MODEL_NAME} Classifier"] = {
    "accuracy": adaboost_accuracy,
    "no_features": 58,
}
print_basic_metrics(model_name=f"{ADABOOST_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

ADA BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.917
Precision (weighted):    0.923
Recall (weighted):       0.917
F1-Score (weighted):     0.917

Confusion Matrix:
-----------------
True Negatives:  507
False Positives: 28
False Negatives: 48
True Positives:  338

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93       535
           1       0.92      0.88      0.90       386

    accuracy                           0.92       921
   macro avg       0.92      0.91      0.91       921
weighted avg       0.92      0.92      0.92       921



#### Search for the Best Combination of Hyperparameters for Ada Boosting Classifier

In [848]:
param_grid = {
    "adaboost_classifier__n_estimators": [50, 100, 200, 300],
    "adaboost_classifier__learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0],
    # "adaboost_classifier__algorithm": ["SAMME", "SAMME.R"],
    "adaboost_classifier__estimator__max_depth": [1, 3, 5, 7],
    "adaboost_classifier__estimator__min_samples_split": [2, 5, 10],
    "adaboost_classifier__estimator__min_samples_leaf": [1, 2, 5],
}

random_adaboost = RandomizedSearchCV(
    adaboost_pipeline,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_adaboost.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._state=777))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'adaboost_classifier__estimator__max_depth': [1, 3, ...], 'adaboost_classifier__estimator__min_samples_leaf': [1, 2, ...], 'adaboost_classifier__e...ator__min_samples_split': [2, 5, ...], 'adaboost_classifier__learning_rate': [0.01, 0.05, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits 

In [849]:
best_adaboost = random_adaboost.best_estimator_  
classifier = best_adaboost.named_steps["adaboost_classifier"]

print("Best Parameters:", random_adaboost.best_params_)
print("Best CV Score:", random_adaboost.best_score_)

y_pred = best_adaboost.predict(X_test)
tuned_adaboost_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(model_name=f"Tuned {ADABOOST_MODEL_NAME} Classifier", y_test=y_test, y_pred=y_pred)

Best Parameters: {'adaboost_classifier__n_estimators': 200, 'adaboost_classifier__learning_rate': 1.0, 'adaboost_classifier__estimator__min_samples_split': 2, 'adaboost_classifier__estimator__min_samples_leaf': 5, 'adaboost_classifier__estimator__max_depth': 5}
Best CV Score: 0.9532608695652174
TUNED ADA BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.957
Precision (weighted):    0.948
Recall (weighted):       0.957
F1-Score (weighted):     0.957

Confusion Matrix:
-----------------
True Negatives:  515
False Positives: 20
False Negatives: 20
True Positives:  366

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.96      0.96       535
           1       0.95      0.95      0.95       386

    accuracy                           0.96       921
   macro avg       0.96      0.96      0.96       921
weighted avg       0.96      0.96      0.96       921



In [850]:
classifier_metrics[f"Tuned {ADABOOST_MODEL_NAME} Classifier"] = {
    "accuracy": tuned_catboost_accuracy,
    "no_features": 58
}

pprint.pprint(classifier_metrics)

{'Ada Boosting Classifier': {'accuracy': 0.9174809989142236, 'no_features': 58},
 'CatBoost Classifier': {'accuracy': 0.9424538545059717, 'no_features': 58},
 'Extreme Gradient Boosting Classifier': {'accuracy': 0.9478827361563518,
                                          'no_features': 58},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
                                      'no_features': 58},
 'Gradient Boosting Classifier': {'accuracy': 0.9391965255157437,
                                  'no_features': 58},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
                                    'no_features': 58},
 'Light Gradient Boosting Classifier': {'accuracy': 0.9500542888165038,
                                        'no_features': 58},
 'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
                                    'no_features': 58},
 'Multilayer Perceptor Classifier': {'accuracy': 0.9239956568946797,
         

#### Optimizing Ada Boosting Classifier

In [851]:
importances = np.abs(classifer.feature_importances_)
feature_importance = pd.Series(importances, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)  
print(feature_importance)

word_freq_george              10.336616
word_freq_hp                   7.270365
capital_run_length_average     6.268333
capital_run_length_longest     5.934324
char_freq_%21                  5.532921
char_freq_%24                  4.427405
word_freq_remove               4.392892
capital_run_length_total       4.247730
word_freq_edu                  3.938186
word_freq_free                 3.164758
word_freq_you                  3.092677
word_freq_our                  3.052979
word_freq_your                 2.964509
word_freq_will                 2.758455
word_freq_meeting              2.640583
char_freq_%28                  2.545807
word_freq_re                   2.445261
word_freq_business             1.811086
word_freq_1999                 1.725523
word_freq_email                1.381197
word_freq_money                1.374051
word_freq_85                   1.264360
word_freq_000                  1.224947
word_freq_technology           1.187199
word_freq_mail                 0.945775


In [852]:
important_features_scores = feature_importance[feature_importance > 3]
important_features = feature_importance[feature_importance > 3].index.to_list()
print(important_features_scores)
print(f"NUMBER OF FEATURES: {len(important_features)}")

word_freq_george              10.336616
word_freq_hp                   7.270365
capital_run_length_average     6.268333
capital_run_length_longest     5.934324
char_freq_%21                  5.532921
char_freq_%24                  4.427405
word_freq_remove               4.392892
capital_run_length_total       4.247730
word_freq_edu                  3.938186
word_freq_free                 3.164758
word_freq_you                  3.092677
word_freq_our                  3.052979
dtype: float64
NUMBER OF FEATURES: 12


In [853]:
X_train_optimized = X_train[important_features]
X_test_optimized = X_test[important_features]

numerical_features = (
    df
    .drop(columns=["class"])
    .loc[:, important_features]
    .select_dtypes(include=np.number)
    .columns
)

categorical_features = (
	df
	.drop(columns=["class"])
	.select_dtypes(include=["string", "object"])
	.columns
)

In [854]:
# Best Parameters: 
# {'adaboost_classifier__n_estimators': 200, 
# 'adaboost_classifier__learning_rate': 1.0, 
# 'adaboost_classifier__estimator__min_samples_split': 2, 
# 'adaboost_classifier__estimator__min_samples_leaf': 5, 
# 'adaboost_classifier__estimator__max_depth': 5}

In [855]:
base_estimator = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=2,
    min_samples_leaf=5,
    random_state=RANDOM_STATE
)

adaboost = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=200,
    learning_rate=1.0,
    # algorithm='SAMME.R',
    random_state=RANDOM_STATE
)

num_pipeline = Pipeline([  
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())  
])  
  
cat_pipeline = Pipeline([  
    ("imputer", SimpleImputer(strategy="most_frequent")),  
    ("encoder", OneHotEncoder(handle_unknown="ignore"))  
])  
  
preprocessor = ColumnTransformer([  
    ("num", num_pipeline, numerical_features),  
    ("cat", cat_pipeline, categorical_features)  
])  
  
adaboost_pipeline = Pipeline([  
    ("preprocessing", preprocessor),  
    ("adaboost_classifier", adaboost)
])

adaboost_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('adaboost_classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

In [856]:
y_pred = adaboost_pipeline.predict(X_test)
adaboost_accuracy = accuracy_score(y_test, y_pred)
print_basic_metrics(
    model_name=f"Tuned and Optimized {ADABOOST_MODEL_NAME} Classifier", 
    y_test=y_test, 
    y_pred=y_pred
)

TUNED AND OPTIMIZED ADA BOOSTING CLASSIFIER PERFORMANCE METRICS
Accuracy:                0.936
Precision (weighted):    0.934
Recall (weighted):       0.936
F1-Score (weighted):     0.936

Confusion Matrix:
-----------------
True Negatives:  510
False Positives: 25
False Negatives: 34
True Positives:  352

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.95      0.95       535
           1       0.93      0.91      0.92       386

    accuracy                           0.94       921
   macro avg       0.94      0.93      0.93       921
weighted avg       0.94      0.94      0.94       921



In [857]:
classifier_metrics[f"Tuned and Optimized {ADABOOST_MODEL_NAME} Classifier"] = {
    "accuracy": adaboost_accuracy,
    "no_features": len(important_features_scores)
}

In [858]:
pprint.pprint(sorted(classifier_metrics.items(), key=lambda stats: stats[1]["accuracy"], reverse=True))

[('Extreme Random Forest Classifier',
  {'accuracy': 0.9554831704668838, 'no_features': 58}),
 ('Tuned Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Light Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Tuned Light Gradient Boosting Classifier',
  {'accuracy': 0.9500542888165038, 'no_features': 58}),
 ('Extreme Gradient Boosting Classifier',
  {'accuracy': 0.9478827361563518, 'no_features': 58}),
 ('Tuned Random Forest Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Gradient Boosting Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned CatBoost Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('Tuned Ada Boosting Classifier',
  {'accuracy': 0.9457111834961998, 'no_features': 58}),
 ('CatBoost Classifier', {'accuracy': 0.9424538545059717, 'no_features': 58}),
 ('Gradient Boosting Classifier',
  {'accuracy': 0.9391965255

#### Endnotes

The Ada Boosting model was examined across three distinct versions, each reflecting a different strategic approach to model configuration. The initial Ada Boosting classifier, operating on fifty-eight features, achieved a baseline accuracy of 0.9175. A subsequent tuned version retained the full feature set of fifty-eight variables and improved performance substantially to 0.9457, representing a significant gain of nearly three percentage points through hyperparameter optimisation. The final iteration, a tuned and optimised Ada Boosting classifier, underwent a reduction in the feature space to twelve predictors, but this simplification came at a considerable cost, as its accuracy declined to 0.9359, falling below the tuned version and only marginally exceeding the original baseline.

This progression offers a clear illustration of the bias-variance tradeoff within the Ada Boosting framework. The tuned Ada Boosting model, leveraging the complete set of fifty-eight features, achieves its highest accuracy by exploiting the full informational richness of the dataset, suggesting that the algorithm's adaptive weighting mechanism effectively controls variance while maintaining low bias. The optimised version, however, deliberately increases bias by pruning to twelve features in an effort to reduce variance and enhance generalisability. The empirical outcome, however, refutes the benefit of this strategy; the accuracy decline of nearly one percentage point from the tuned version indicates that the discarded features contained predictive value that could not be compensated by the remaining variables. The reduction in complexity was therefore not justified, as the simplified model failed to deliver competitive performance and instead produced a result inferior to several other ensemble methods.

When comparing the best-performing Ada Boosting version, the tuned classifier at 0.9457, against the broader experimental suite, it occupies a solid mid-tier position among the top performers. It ties with the tuned random forest, tuned gradient boosting, and tuned CatBoost at the same accuracy, while falling behind the extreme random forest at 0.9555 and the light gradient boosting variants at 0.9501. The tuned Ada Boosting comfortably outperforms the extreme gradient boosting baseline at 0.9479, all multilayer perceptron variants, and all K-nearest neighbours and logistic regression classifiers. Within its own family, the tuned version is unequivocally superior to both the baseline and the optimised iteration, with the latter still exceeding several simpler models but remaining the weakest among the Ada Boosting lineage. This hierarchy underscores that for Ada Boosting, preserving feature richness while fine-tuning hyperparameters yields the most robust predictive performance, and any aggressive reduction in complexity proves detrimental to the model's effectiveness.

### Conclusion

In [859]:
classifier_metrics

{'Logistic Regression Classifier': {'accuracy': 0.9142236699239956,
  'no_features': 58},
 'Tuned Logistic Regression Classifier': {'accuracy': 0.9131378935939196,
  'no_features': 58},
 'Tuned and Optimized Logistic Regression Classifier': {'accuracy': 0.8631921824104235,
  'no_features': 12},
 'K-Nearest Neighbors Classifier': {'accuracy': 0.8935939196525515,
  'no_features': 58},
 'Tuned K-Nearest Neighbors Classifier': {'accuracy': 0.9153094462540716,
  'no_features': 58},
 'Random Forest Classifier': {'accuracy': 0.9326818675352877,
  'no_features': 58},
 'Tuned Random Forest Classifier': {'accuracy': 0.9457111834961998,
  'no_features': 58},
 'Tuned and Optimized Random Forest Classifier': {'accuracy': 0.9305103148751357,
  'no_features': 12},
 'Extreme Random Forest Classifier': {'accuracy': 0.9554831704668838,
  'no_features': 58},
 'Tuned Extreme Random Forest': {'accuracy': 0.9337676438653637,
  'no_features': 58},
 'Tuned and Optimized Extreme Random Forest Classifier': {'ac

After an exhaustive comparative analysis across eight distinct algorithmic families, encompassing both linear and ensemble methodologies, we arrive at a definitive conclusion regarding the optimal predictive strategy for this classification task. The experimental design, which systematically evaluated baseline, tuned, and feature-optimised versions, has yielded a clear hierarchy of performance that offers profound insights into the bias-variance tradeoff in practice.

The Extreme Random Forest Classifier, in its original configuration with fifty-eight features, stands unchallenged as the supreme performer, achieving an accuracy of 0.9555. This result is particularly striking as it outperforms its own tuned and optimised counterparts, which suffered a notable performance degradation to 0.9338 and 0.9273 respectively. This phenomenon underscores a critical lesson in applied machine learning: for highly randomised ensemble methods, the inherent variance reduction and feature randomness already provide robust generalisation, rendering additional hyperparameter tuning and aggressive feature pruning not only unnecessary but actively harmful.

Conversely, the Linear models exhibited the most pronounced sensitivity to feature reduction. The logistic regression family demonstrated a steep decline from 0.9142 to 0.8632 upon pruning to twelve features, confirming that simpler parametric models rely heavily on the full informational breadth of the input space. Meanwhile, the Boosting families, including Gradient, Extreme Gradient, Light Gradient, and CatBoost, all reached a performance plateau near 0.9457 to 0.9501 when operating on the full set of predictors, with Light Gradient Boosting and Extreme Gradient Boosting tying at 0.9501, just marginally behind the Extreme Random Forest. However, every optimised boosting variant suffered a substantial accuracy drop, reinforcing the principle that these powerful sequential models already incorporate effective regularisation and feature importance mechanisms, making post-hoc feature selection largely redundant.

In summary, the empirical evidence advocates strongly for the adoption of the Extreme Random Forest Classifier as the final production model. Its superior accuracy, combined with the fact that its best performance was achieved without costly hyperparameter optimisation or feature engineering, makes it the most robust and efficient choice. The data decisively demonstrates that, within this domain, preserving the richness of the feature space and leveraging the inherent strengths of high-variance, low-bias ensembles are the keys to achieving state-of-the-art predictive performance.